# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/h0pi/flyrank-ml-intenrship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

The paper (“The State of AI-Driven SEO in Numbers”, March 2026) is written for a broad audience and is open about its own standards: it labels the ML pages as exploratory, flags small samples (the 361+ freshness bucket) and lists its limitations. The questions below are the ones I would want someone to ask about my own work.

### Finding A: “The Freshness Multiplier” (Finding #4 and the Content Refresh Playbook)
**Claim:** pages older than 365 days that were refreshed in the last 30 days show a 3.2× higher health score and 57× more impressions than old pages that were not refreshed, and refresh timing is called “one of the strongest measured levers”.

**Where the comparison comes from:** “refreshed” versus “not refreshed” is not an experiment. Someone *chose* which old pages to refresh, and a team usually refreshes pages that still have demand.

**My questions:**
1. **Selection:** were the refreshed pages already stronger *before* the refresh? Comparing the same pages before and after the update, or against similar untouched pages matched on their earlier impressions, would separate “refresh helps” from “we refresh the pages that were doing well anyway”.
2. **Timing of the measurement:** are the 90-day impressions measured *after* the refresh date, or does the window include the weeks before it? If the window straddles the update, part of the gap was there before the refresh.
3. **Survivor sample:** the active-content subset keeps only pages with impressions > 0 and sessions > 0, so old pages that died completely drop out of the “untouched” group. How does the gap change on the full inventory?

The paper itself says correlations do not prove causation. My suggestion would be to phrase the playbook step the same way, e.g. “refreshed mature pages are *associated with* much higher visibility”, until a before/after comparison is available.

### Finding B: “What Predicts Growth?” (ML appendix, logistic regression, 71% holdout accuracy)
**Claim:** a logistic regression separates growing from declining pages with 71% holdout accuracy. Content age is the strongest negative signal, and days visible and recent impressions are strong positive signals.

**Where the label comes from:** trend direction = last-30-days vs previous-30-days impressions (±10%).

**My questions:**
1. **Base rate:** Finding #1 reports 74,187 rising vs 45,272 falling pages, so always predicting “growing” already scores about 62% (computed below). 71% is about 9 points of skill over that. Reporting the base rate (or a ranking metric like ROC-AUC) next to the accuracy would make the number readable.
2. **Overlapping windows:** “recent impressions” and “days visible” are measured over windows that include the last 30 days, and the label is computed from those same last 30 days. A page that grew in the last 30 days automatically has more recent impressions. Are the features restricted to the *previous* 30 days only?
3. **Split design:** the methodology lists an 80/20 split, presumably random across pages. Pages of the same brand share site-wide trends, so the test set contains sister pages of the training set. A split grouped by brand would show how much of the 71% carries over to a brand the model has never seen. Section 2 measures exactly this effect on my own model.

In [1]:

# Base rate for Finding B, from the paper's own counts (Finding #1: 74,187 rising vs 45,272 falling pages)
rising, falling = 74_187, 45_272
majority = rising / (rising + falling)
print(f"always predicting 'growing' would score {majority:.1%} accuracy")
print(f"reported holdout accuracy 71% -> about {71 - majority * 100:.0f} percentage points above that naive baseline")


always predicting 'growing' would score 62.1% accuracy
reported holdout accuracy 71% -> about 9 percentage points above that naive baseline


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Same data, label and features as weeks 3–5 (March 2026, decision moment 16 March; inputs only from 1–15 March). The model is the week-5 pick, Logistic Regression.

- **Before:** 5-fold *random* split over pages. This is the kind of split the paper's appendix describes.
- **After:** 5-fold split *grouped by client*. Every test fold contains only clients the model never saw.

In [2]:

%pip -q install duckdb huggingface_hub scikit-learn

import os, json, getpass
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.inspection import permutation_importance

SEED = 42
pd.set_option("display.width", 180)

def hf_token():
    tok = os.environ.get("HF_TOKEN")
    if not tok:
        try:
            from google.colab import userdata
            tok = userdata.get("HF_TOKEN")
        except Exception:
            pass
    if not tok:
        try:
            from huggingface_hub import get_token
            tok = get_token()
        except Exception:
            pass
    return tok or getpass.getpass("Hugging Face READ token: ")

ROOT = Path.cwd().parents[1] if Path.cwd().name == "notebooks" else Path.cwd()
OUT = ROOT / "work" / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
CACHE = OUT / "march_pages.parquet"          # same cache file as week 4

if not CACHE.exists():
    con = duckdb.connect()
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token()}')")
    REL = "hf://datasets/FlyRank/internship-warehouse"
    con.sql(f"""
        WITH d AS (
            SELECT client_hash_id, content_hash_id, report_date, gsc_impressions, gsc_clicks, gsc_avg_position
            FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
            WHERE gsc_data_available IS TRUE
        ),
        p AS (
            SELECT client_hash_id, content_hash_id,
                   SUM(gsc_impressions) FILTER (WHERE report_date <= DATE '2026-03-15')                            AS imp_h1,
                   SUM(gsc_clicks)      FILTER (WHERE report_date <= DATE '2026-03-15')                            AS clk_h1,
                   SUM(gsc_impressions) FILTER (WHERE report_date <= DATE '2026-03-07')                            AS imp_w1,
                   SUM(gsc_impressions) FILTER (WHERE report_date BETWEEN DATE '2026-03-08' AND DATE '2026-03-15') AS imp_w2,
                   SUM(gsc_avg_position * gsc_impressions)
                       FILTER (WHERE report_date <= DATE '2026-03-15' AND gsc_impressions > 0 AND gsc_avg_position > 0) AS pos_x_imp,
                   SUM(gsc_impressions)
                       FILTER (WHERE report_date <= DATE '2026-03-15' AND gsc_impressions > 0 AND gsc_avg_position > 0) AS imp_with_pos,
                   SUM(gsc_impressions) FILTER (WHERE report_date >= DATE '2026-03-16')                            AS imp_h2
            FROM d GROUP BY 1, 2
            HAVING SUM(gsc_impressions) FILTER (WHERE report_date <= DATE '2026-03-15') >= 100
        )
        SELECT p.*, c.content_created_date, c.content_type
        FROM p JOIN read_parquet('{REL}/dim_content.parquet') c USING (client_hash_id, content_hash_id)
        WHERE c.content_created_date < DATE '2026-03-01'
    """).df().to_parquet(CACHE)

df = pd.read_parquet(CACHE).sort_values(["client_hash_id", "content_hash_id"]).reset_index(drop=True)
df["imp_h2"] = df["imp_h2"].fillna(0)
# ---- inputs: all knowable on 2026-03-16 ----
df["imp_per_day"]   = df["imp_h1"] / 15
df["log_imp"]       = np.log1p(df["imp_per_day"])
df["momentum"]      = np.log((df["imp_w2"].fillna(0) / 8 + 1) / (df["imp_w1"].fillna(0) / 7 + 1))
df["ctr"]           = df["clk_h1"] / df["imp_h1"]
df["avg_position"]  = df["pos_x_imp"] / df["imp_with_pos"]
df["age_days"]      = (pd.Timestamp("2026-03-15") - pd.to_datetime(df["content_created_date"])).dt.days
df["is_feedly"]     = (df["content_type"] == "feedly article").astype(int)
df["is_comparison"] = (df["content_type"] == "comparison article").astype(int)
# ---- label: 16-31 March, used only as the target ----
df["is_declining"]  = ((df["imp_h2"] / 16) < 0.8 * df["imp_per_day"]).astype(int)

FEATURES = ["log_imp", "momentum", "ctr", "avg_position", "age_days", "is_feedly", "is_comparison"]
assert df[FEATURES].notna().all().all()
# week-4 baseline rule, frozen exactly as it was
df["baseline_score"] = np.where(df["momentum"] < -0.1, -df["momentum"] * np.log1p(df["imp_per_day"]), 0.0)

print(f"pages: {len(df):,} | clients: {df.client_hash_id.nunique()} | base rate: {df.is_declining.mean():.3f}")
print(f"scikit-learn {sklearn.__version__} | seed {SEED}")


Note: you may need to restart the kernel to use updated packages.


pages: 74,377 | clients: 36 | base rate: 0.336
scikit-learn 1.9.1 | seed 42


In [3]:

from sklearn.model_selection import KFold

def lr():
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))

def out_of_fold(splitter, groups=None, feats=FEATURES):
    oof = np.zeros(len(df))
    for tr, te in splitter.split(df, df["is_declining"], groups):
        oof[te] = lr().fit(df.loc[tr, feats], df.loc[tr, "is_declining"]).predict_proba(df.loc[te, feats])[:, 1]
    return oof

def summarise(scores):
    order = np.argsort(-scores)
    y = df["is_declining"].values
    return {"ROC-AUC": roc_auc_score(y, scores), **{f"P@{k}": y[order[:k]].mean() for k in (50, 500, 1000, 5000)}}

df["oof_random"]  = out_of_fold(KFold(n_splits=5, shuffle=True, random_state=SEED))
df["oof_grouped"] = out_of_fold(GroupKFold(n_splits=5), groups=df["client_hash_id"])

before_after = pd.DataFrame({
    "BEFORE: random split": summarise(df["oof_random"].values),
    "AFTER: grouped by client": summarise(df["oof_grouped"].values),
    "Week-4 rule (no training)": summarise(df["baseline_score"].values),
}).T
before_after.loc["Base rate"] = [0.5] + [df["is_declining"].mean()] * 4
before_after["ROC-AUC gap vs random"] = before_after["ROC-AUC"] - before_after.loc["BEFORE: random split", "ROC-AUC"]
before_after.round(3)


,ROC-AUC,P@50,P@500,P@1000,P@5000,ROC-AUC gap vs random
BEFORE: random split,0.697,0.980,0.892,0.835,0.711,0.000
AFTER: grouped by client,0.658,0.980,0.880,0.825,0.663,-0.039
Week-4 rule (no training),0.633,0.980,0.832,0.779,0.639,-0.064
Base rate,0.500,0.336,0.336,0.336,0.336,-0.197


In [4]:

# Why the gap: clients differ a lot in how often their pages decline
per_client = (df.groupby("client_hash_id")
                .agg(pages=("is_declining", "size"), decline_rate=("is_declining", "mean"),
                     mean_age_days=("age_days", "mean"), mean_ctr=("ctr", "mean"))
                .sort_values("pages", ascending=False).head(8))
per_client.index = [f"client {i + 1}" for i in range(len(per_client))]   # pseudonyms not needed here
per_client.round(3)


,pages,decline_rate,mean_age_days,mean_ctr
client 1,17562,0.350,238.194,0.002
client 2,13889,0.431,162.301,0.002
client 3,10215,0.426,133.830,0.003
client 4,7477,0.098,316.326,0.002
client 5,6305,0.202,148.164,0.003
client 6,3925,0.154,229.387,0.004
client 7,2811,0.313,86.454,0.003
client 8,2261,0.313,48.208,0.005


**What the before/after shows:**

- **The random split looks better than it is.** ROC-AUC drops by about 0.04 (≈ 0.70 → ≈ 0.66) when test clients are unseen, and P@500 and P@1000 also drop a little.
- **Why it happens:** the model has no client-ID input, so the gap comes from memorising client character through the features. Large clients have very different decline rates (about 10% for one, about 43% for another, table above) and different typical ages and CTRs. In a random split, the model learns e.g. "pages with this kind of age and CTR decline often", which is really "pages of this client decline often", and gets rewarded because sister pages sit in the test set.
- **The grouped number is the honest one** for the real use case (scoring a client's pages the model hasn't been tuned on), and it is the number I keep. It still beats the week-4 rule (≈ 0.66 vs ≈ 0.63 ROC-AUC; 0.88 vs 0.83 at P@500).
- **One more honest correction from earlier weeks:** in week 3 I reported 0.721 ROC-AUC from *one* grouped split. Five grouped folds give ≈ 0.67 on average, with a fold-to-fold spread of about ±0.05. A single split can land on a lucky set of clients.
- **Not yet tested:** a *time-aware* split (train on one month, test on the next). All rows come from March 2026, so I cannot yet say the model holds in a different month.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

**Timeline:** decision moment = morning of 16 March 2026. The label uses 16–31 March.

| Feature | Built from | Known on 16 March? |
|---|---|---|
| `log_imp` | impressions 1–15 March | yes |
| `momentum` | impressions 1–7 vs 8–15 March | yes, ends 15 March |
| `ctr` | clicks ÷ impressions 1–15 March | yes |
| `avg_position` | position 1–15 March (impression-weighted, zero positions excluded) | yes |
| `age_days` | creation date → 15 March | yes, fixed at creation |
| `is_feedly`, `is_comparison` | content type | yes, fixed at creation |
| *excluded* `content_updated_date`, `last_optimized_date` | July 2026 snapshot | **no**: could contain edits made after 16 March |
| *excluded* `client_hash_id` | pseudonym | used only for grouping the split |

Below, the checks from the leakage checklist.

In [5]:

# (a) No single feature is "too good" on its own (a leak usually shows up as one feature near 1.0)
single = {f: max(roc_auc_score(df["is_declining"], df[f]), 1 - roc_auc_score(df["is_declining"], df[f])) for f in FEATURES}
display(pd.Series(single, name="single-feature ROC-AUC").sort_values(ascending=False).round(3).to_frame())

# (b) The harness can detect a leak: add a label-window column on purpose -> score must jump
# the label is a ratio of late-March to early-March impressions, so the leak is that same ratio
df["LEAK_imp_h2_per_day"] = np.log((df["imp_h2"] / 16 + 1) / (df["imp_per_day"] + 1))
leaky = summarise(out_of_fold(GroupKFold(n_splits=5), df["client_hash_id"], FEATURES + ["LEAK_imp_h2_per_day"]))
print(f"grouped ROC-AUC honest: {summarise(df['oof_grouped'].values)['ROC-AUC']:.3f} | "
      f"with deliberate leak: {leaky['ROC-AUC']:.3f}  <- invalid, removed")
df = df.drop(columns=["LEAK_imp_h2_per_day"])

# (c) No product flags / existing-system scores anywhere in the frame
PRODUCT_FLAGS = {"health_score", "priority_score", "action_type", "needs_ctr_fix", "is_quick_win", "refresh_tier", "trend_direction", "trend_pct"}
assert not PRODUCT_FLAGS & set(df.columns)
print("product flags / trend fields in the frame:", sorted(PRODUCT_FLAGS & set(df.columns)) or "none")

# (d) Population selection: which filters could carry outcome-window information?
print(f"filters: >=100 impressions on 1-15 Mar, created before March, GSC data available -> {len(df):,} pages")
print(f"pages with zero impressions on 16-31 Mar (vanished from search or tracking): {(df.imp_h2 == 0).sum()} "
      f"({(df.imp_h2 == 0).mean():.2%}) -> counted as 'declining'")


,single-feature ROC-AUC
momentum,0.654
ctr,0.580
age_days,0.557
avg_position,0.530
log_imp,0.505
is_comparison,0.504
is_feedly,0.502


grouped ROC-AUC honest: 0.658 | with deliberate leak: 1.000  <- invalid, removed
product flags / trend fields in the frame: none
filters: >=100 impressions on 1-15 Mar, created before March, GSC data available -> 74,377 pages
pages with zero impressions on 16-31 Mar (vanished from search or tracking): 131 (0.18%) -> counted as 'declining'


**Leakage verdict:**

- **(a)** The strongest single input is momentum, and it is nowhere near "too good". No feature reads the answer.
- **(b)** The harness works: one column from the label window pushes the score close to 1.0, and removing it brings back the honest number.
- **(c)** No product flags or trend fields are used.
- **(d)** Population: every filter uses only 1–15 March or fixed data, so selecting pages does not peek at the outcome. One choice to disclose: the 131 pages (0.18%) that got *no* impressions at all in late March count as "declining". Some of them may have been removed or lost tracking rather than decayed, so that is a label-definition choice, not leakage.

### Real failure examples (honest grouped scores)

In [6]:

col = "oof_grouped"
show = ["content_type", "imp_per_day", "momentum", "ctr", "avg_position", "age_days", col, "is_declining"]
fp = df[df.is_declining == 0].nlargest(3, col)      # "will decline" with high confidence, but it didn't
fn = df[df.is_declining == 1].nsmallest(2, col)     # "safe" with high confidence, but it declined
cases = pd.concat([fp, fn])[show].copy()
cases.insert(0, "type", ["false alarm"] * 3 + ["missed decline"] * 2)
cases["imp/day 16-31 Mar"] = (df.loc[cases.index, "imp_h2"] / 16).round(1)
cases.round(3)


,type,content_type,imp_per_day,momentum,ctr,avg_position,age_days,oof_grouped,is_declining,imp/day 16-31 Mar
8870,false alarm,keyword article,8.200,-2.683,0.000,48.374,26,0.970,0,25.0
66719,false alarm,keyword article,20.133,-2.832,0.000,3.253,68,0.955,0,30.2
66830,false alarm,keyword article,28.667,-2.873,0.000,45.998,54,0.954,0,26.8
52772,missed decline,keyword article,358.067,6.511,0.002,6.353,17,0.000,1,228.8
53896,missed decline,keyword article,207.467,5.966,0.003,12.206,17,0.001,1,145.6


- **False alarms:** pages with a sharp week-1 → week-2 drop that recovered in late March (to ~25–30 impressions/day). With only 15 days of history, a one-week dip looks the same as the start of a real decline. All three are fairly young (26–68 days) with low daily volume (8–29 impressions/day), where a handful of impressions moves the momentum a lot.
- **Missed declines:** very young pages that were *rising* fast in early March and then settled. The model reads growth as "safe", but a launch surge that levels off counts as a decline under my label. That is a label-definition limit as much as a model error.

What would reduce these errors: a minimum impression floor for momentum, an age gate for pages younger than about 30 days, and more than one month of history.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

| My earlier wording | Rewritten | Why |
|---|---|---|
| “Logistic Regression **beats the baseline rule** where it matters” (week 5) | “On March 2026 data, grouped by client, the logistic-regression ranking **measured** higher precision than the week-4 rule at K = 500 and 1,000 (≈ 0.88 vs 0.83, ≈ 0.83 vs 0.78) and **tied** at K = 50.” | One month, one dataset. The result is measured, not general. |
| “A page that was already sliding is **more likely to keep sliding**” (week 4) | “Pages whose impressions fell from week 1 to week 2 of March **were observed** to decline in late March more often (54% vs 21% for pages rising strongly). This is a **directional** signal, not a cause.” | It is an association inside one month. |
| “**Older pages decline less**, the opposite of the refresh-flag assumption” (week 5) | “In this slice, page age **had a small negative weight** and older pages **were observed** to decline less often. This **does not test** whether refreshing a page changes its future.” | It doesn't contradict the paper's refresh finding, which is about refreshed vs untouched pages. |
| “The model ranks pages for review **measurably better**” | “The score is **decision-support** for ordering a review queue. A person still checks each page, and cliff drops and very new pages need a separate look.” | Keeps the human in the loop and names the known failure modes. |
| Week 3: “0.721 ROC-AUC” | “≈ 0.67 ROC-AUC averaged over 5 client-grouped folds (range ≈ 0.59–0.74). The earlier 0.721 came from a single split.” | Single-split numbers overstate stability. |

**My capstone claim, in one sentence:** *On March 2026 pages from 36 clients, a simple logistic regression using only first-half-of-month signals measured a modest, directional improvement over a transparent rule for ordering a content-review queue on clients it had not seen. It is decision-support for prioritising human review, not a prediction of Google's behaviour.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.